# 9-element UniversalXAS / Tuned-UniversalXAS sweep (all-in-one notebook)

Runs where the ZN notebook already exported 64-D features for all 9 elements
(Ti, V, Cr, Mn, Fe, Co, Ni, Cu, Zn).

Design:
- Stage 1: 100 UniversalXAS trials with the published hidden dimensions.
- Stage 2: 300 tuned trials per element from the best Universal checkpoint.
- Select each element's checkpoint and hyperparameters by its validation median MSE.
- Trial 0 is the unchanged Universal checkpoint. Retain it if no trial improves validation.
- Use normal fine-tuning, without epoch rollback. Loss normalization affects training only.
- Hidden dimensions are fixed at the established UniversalXAS architecture everywhere.
- Stage 3: check all selected validation scores, then evaluate test once per selected head.

Features come from run `zn_full`. The sweep varies learning rate, schedule, dropout, balance, and normalization.
Reruns overwrite checkpoints and result files in the configured `RUN_NAME` folder.
The saved summary and hyperparameter table contain one selected record per element.
The final table includes the selected settings.
Zn gets two 2x2 percentile showcase figures for validation and test.


In [ ]:
import json as _json, math, random
from hashlib import file_digest
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from omnixas.model.xasblock import XASBlock

ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn']
TASKS = [e + '_FEFF' for e in ELEMENTS]
DIM = 64; SPEC = 141
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Auto-detect the 9-element features exported by the ZN notebook.
# (train_zn_expert_and_full.ipynb): OUTPUT_ROOT = repo / output/training/zn_expert_study,
# run name = "zn_full", features = run / "features".
REPO_ROOT = Path.cwd()
for p in (Path.cwd(), *Path.cwd().parents):
    if (p / 'omnixas').is_dir():
        REPO_ROOT = p
        break
candidates = [
    REPO_ROOT / 'output/training/zn_expert_study/zn_full/features',
    Path('/mnt/c/Users/anton/desktop/omnixas/output/training/zn_expert_study/zn_full/features'),
    Path.home() / 'omnixas/output/training/zn_expert_study/zn_full/features',
]
FEATURES_DIR = None
for c in candidates:
    if (c / 'Ti_FEFF_train_X.txt').is_file():
        FEATURES_DIR = c
        break
if FEATURES_DIR is None:
    raise FileNotFoundError(
        'Could not find zn_full features. Set FEATURES_DIR manually in cell 1.'
    )
print('repo root:', REPO_ROOT)
print('features dir:', FEATURES_DIR)

RUN_NAME = 'zn_universal_tuned_per_element'  # Reruns overwrite this run's results.
OUTPUT = REPO_ROOT / 'output/training/zn_sweep' / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print('output (overwrite enabled):', OUTPUT)
FEATURE_HASHES = {}

def load(task, split):
    x = np.atleast_2d(np.loadtxt(FEATURES_DIR / (task + '_' + split + '_X.txt'), dtype=np.float32))
    y = np.atleast_2d(np.loadtxt(FEATURES_DIR / (task + '_' + split + '_y.txt'), dtype=np.float32))
    if x.shape != (len(y), DIM) or y.shape[1] != SPEC or not len(y):
        raise ValueError(f'Invalid feature/target shapes for {task}/{split}: {x.shape}, {y.shape}')
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError(f'Non-finite data for {task}/{split}')
    for suffix in ('X', 'y'):
        path = FEATURES_DIR / f'{task}_{split}_{suffix}.txt'
        with path.open('rb') as handle:
            FEATURE_HASHES[path.name] = file_digest(handle, 'sha256').hexdigest()
    return x, y


In [ ]:
def make_head(hidden_dims, dropout, state=None):
    with patch.object(XASBlock, 'DROPOUT', dropout):
        head = XASBlock(DIM, list(hidden_dims), SPEC).to(DEVICE)
    if state is not None:
        head.load_state_dict(state, strict=True)
    return head


def copy_state(head):
    return {name: value.detach().cpu().clone() for name, value in head.state_dict().items()}


def predict_spectra(head, X, batch_size):
    head.eval()
    device = next(head.parameters()).device
    Xt = torch.as_tensor(X, dtype=torch.float32, device=device)
    with torch.inference_mode():
        return torch.cat([head(b).cpu() for b in Xt.split(batch_size)]).numpy()


def prediction_errors(head, X, y, batch_size):
    err = ((predict_spectra(head, X, batch_size) - y) ** 2).mean(1)
    if not len(err) or not np.isfinite(err).all():
        raise FloatingPointError('Empty or non-finite prediction errors')
    return err


def median_mse(head, X, y, batch_size):
    return float(np.median(prediction_errors(head, X, y, batch_size)))


def train_head(X, y, vX, vY, source, args, *, lr, epochs, schedule, label,
               hidden_dims, dropout=0.05, loss_balance_power=0.0,
               train_task=None, task_base=None, loss_scale=None, seed=None):
    if len(X) < 2 or args.batch_size < 2:
        raise ValueError('BatchNorm training needs at least two rows and batch_size >= 2')
    if epochs < 0 or (source is None and epochs == 0):
        raise ValueError('A fresh head needs at least one training epoch')
    if loss_scale is not None and (not math.isfinite(loss_scale) or loss_scale <= 0):
        raise ValueError('loss_scale must be finite and positive')
    seed = args.seed if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
    head = make_head(hidden_dims, dropout, source)
    Xt = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(y, dtype=torch.float32, device=DEVICE)
    vXt = torch.as_tensor(vX, dtype=torch.float32, device=DEVICE)
    tt = None if train_task is None else torch.as_tensor(train_task, dtype=torch.long, device=DEVICE)
    tb = None if task_base is None else torch.as_tensor(task_base, dtype=torch.float32, device=DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr, fused=DEVICE.type == 'cuda')
    if schedule == 'plateau':
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.5, patience=16, min_lr=1e-6)
    elif schedule == 'cosine':
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=500, eta_min=1e-6)
    else:
        raise ValueError(f'Unknown schedule: {schedule}')

    best, stale, best_epoch, epochs_run = float('inf'), 0, -1, 0
    best_state, source_metric = None, None
    if source is not None:
        source_metric = median_mse(head, vXt, vY, args.batch_size)
        best, best_state = source_metric, copy_state(head)

    for epoch in tqdm(range(epochs), desc=label, unit='epoch', leave=False):
        head.train()
        batches = list(torch.randperm(len(Xt), device=DEVICE).split(args.batch_size))
        if len(batches[-1]) == 1:
            batches[-2] = torch.cat((batches[-2], batches[-1]))
            batches.pop()
        for b in batches:
            opt.zero_grad(set_to_none=True)
            rm = (head(Xt[b]) - yt[b]).square().mean(1)
            if loss_balance_power:
                loss = torch.stack([
                    rm[tt[b] == t].mean() / tb[t].clamp_min(1e-12).pow(loss_balance_power)
                    for t in torch.unique(tt[b])
                ]).mean()
            else:
                loss = rm.mean()
            if loss_scale is not None:
                loss = loss / loss_scale
            loss.backward()
            opt.step()

        metric = median_mse(head, vXt, vY, args.batch_size)
        epochs_run = epoch + 1
        if metric < best:
            best, stale, best_epoch = metric, 0, epoch
            best_state = copy_state(head)
        else:
            stale += 1
        # Continue from the latest weights and Adam state, even after a worse epoch.
        sched.step(metric) if schedule == 'plateau' else sched.step()
        if stale >= args.head_patience:
            break
    return best_state, best, {
        'best_val_mse': best, 'best_epoch': best_epoch, 'epochs_run': epochs_run,
        'source_val_median_mse': source_metric,
        'selection': 'source_checkpoint' if best_epoch == -1 else 'validation_improvement',
        'training_policy': 'standard_with_source_candidate' if source is not None else 'standard',
        'hidden_dims': list(hidden_dims), 'dropout': dropout,
    }


def check_tuned_source_checkpoint():
    """Optional CPU regression check. Call this function before the sweep."""
    rng = np.random.default_rng(7)
    X = rng.normal(size=(5, DIM)).astype(np.float32)
    vX = rng.normal(loc=2, size=(4, DIM)).astype(np.float32)
    check_args = SimpleNamespace(seed=7, batch_size=4, head_patience=60)
    with patch.dict(globals(), DEVICE=torch.device('cpu')):
        head = make_head([8], 0.1).eval()
        source = copy_state(head)
        with torch.inference_mode():
            vY = head(torch.from_numpy(vX)).numpy()
        for epochs in (0, 2):
            state, metric, meta = train_head(
                X, np.zeros((len(X), SPEC), dtype=np.float32), vX, vY, source, check_args,
                lr=0.001, epochs=epochs, schedule='plateau', label='CPU check',
                hidden_dims=[8], dropout=0.1)
            assert metric == 0.0 and meta['selection'] == 'source_checkpoint'
            assert all(torch.equal(state[k], source[k]) for k in source)
            assert all(torch.equal(head.state_dict()[k], source[k]) for k in source)
    print('CPU check passed: source weights and BatchNorm buffers are unchanged.')


In [ ]:
def balanced(train, val, seed):
    n = max(len(X) for X, _ in train.values()); rng = np.random.default_rng(seed)
    xs, ys, ts = [], [], []
    for i, t in enumerate(TASKS):
        X, y = train[t]; ii = rng.integers(len(X), size=n)
        xs.append(X[ii]); ys.append(y[ii]); ts.append(np.full(n, i))
    vx = np.concatenate([val[t][0] for t in TASKS]); vy = np.concatenate([val[t][1] for t in TASKS])
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(ts), vx, vy

train = {t: load(t, 'train') for t in TASKS}
val = {t: load(t, 'val') for t in TASKS}
train_base = np.array([
    float(np.median(((train[t][1] - train[t][1].mean(0)) ** 2).mean(1))) for t in TASKS
])
print('train bases:', dict(zip(ELEMENTS, [round(float(b), 6) for b in train_base])))

# Established UniversalXAS architecture. Do not change this during the sweep.
UNIVERSAL_HIDDEN_DIMS = (500, 500, 550)
DROPOUTS = [0.0, 0.05, 0.10, 0.20]; LBPS = [0.0, 0.5, 1.0, 1.5, 2.0]
args = SimpleNamespace(seed=42, batch_size=4096, head_patience=60,
                       universal_epochs=800, tuned_epochs=1000,
                       universal_trials=100, tuned_trials=300)
if args.universal_trials < 1:
    raise ValueError('universal_trials must be positive')
rng = np.random.default_rng(args.seed + 1000)

ux, uy, utt, uvx, uvy = balanced(train, val, args.seed)
ubest = None; ures = []
for r in range(1, args.universal_trials + 1):
    h = {
        'lr': float(np.exp(rng.uniform(math.log(1e-5), math.log(1e-2)))),
        'hidden_dims': list(UNIVERSAL_HIDDEN_DIMS),
        'schedule': ['plateau', 'cosine'][int(rng.integers(2))],
        'dropout': float(DROPOUTS[int(rng.integers(len(DROPOUTS)))]),
        'loss_balance_power': float(LBPS[int(rng.integers(len(LBPS)))]),
        'seed': int(rng.integers(1_000_000)),
    }
    st, m, meta = train_head(ux, uy, uvx, uvy, None, args,
                             lr=h['lr'], epochs=args.universal_epochs, schedule=h['schedule'],
                             label=f'Universal {r}/{args.universal_trials}',
                             hidden_dims=h['hidden_dims'], dropout=h['dropout'],
                             loss_balance_power=h['loss_balance_power'],
                             train_task=utt, task_base=train_base, seed=h['seed'])
    ures.append({'run': r, **h, 'val_median_mse': float(m), **meta})
    if ubest is None or m < ubest[0]: ubest = (m, st, meta, {'run': r, **h})
    lr_v = h['lr']; hd_v = h['hidden_dims']
    if r == 1 or r % 10 == 0:
        print(f'universal {r}/{args.universal_trials} val_median_mse={m:.4e} '
              f'lr={lr_v:.2e} hidden={hd_v}', flush=True)

um, ust, umeta, uhyp = ubest
torch.save(ust, OUTPUT / 'best_universal.pt')
pd.DataFrame(ures).to_csv(OUTPUT / 'universal_records.csv', index=False)
lr_u = uhyp['lr']; hd_u = uhyp['hidden_dims']; do_u = uhyp['dropout']; lbp_u = uhyp['loss_balance_power']; sd_u = uhyp['seed']
print('BEST UNIVERSAL val_median_mse=' + f'{um:.4e}' + ' lr=' + f'{lr_u:.2e}' + ' hidden=' + str(hd_u) + ' dropout=' + str(do_u) + ' lbp=' + str(lbp_u) + ' seed=' + str(sd_u), flush=True)


In [ ]:
if args.tuned_trials < 1:
    raise ValueError('tuned_trials must be positive')
rng = np.random.default_rng(args.seed + 2000)
best_tuned, tres = {}, []
hdim = list(UNIVERSAL_HIDDEN_DIMS)
if umeta['hidden_dims'] != hdim:
    raise RuntimeError(
        f'Universal hidden dimensions changed: expected {hdim}, got {umeta["hidden_dims"]}'
    )
hdrop = umeta['dropout']
universal_head = make_head(hdim, hdrop, ust)

for t in TASKS:
    trX, trY = train[t]
    vaX, vaY = val[t]
    source_metric = median_mse(universal_head, vaX, vaY, args.batch_size)
    # Reset the winner for EACH element. Trial 0 is the unchanged source.
    best_metric, best_state = source_metric, ust
    best_record = {
        'element': t, 'run': 0, 'lr': None, 'schedule': None, 'loss_normalize': None,
        'seed': None, 'val_median_mse': source_metric, 'best_val_mse': source_metric,
        'source_val_median_mse': source_metric, 'best_epoch': -1, 'epochs_run': 0,
        'selection': 'source_checkpoint', 'training_policy': 'standard_with_source_candidate',
        'hidden_dims': hdim, 'dropout': hdrop,
    }
    tres.append(best_record.copy())
    for r in range(1, args.tuned_trials + 1):
        hp = {
            'lr': float(np.exp(rng.uniform(math.log(1e-5), math.log(5e-3)))),
            'schedule': ['plateau', 'cosine'][int(rng.integers(2))],
            'dropout': float(DROPOUTS[int(rng.integers(len(DROPOUTS)))]),
            'loss_normalize': bool(rng.integers(2)),
            'seed': int(rng.integers(1_000_000)),
        }
        ls = max(float(train_base[TASKS.index(t)]), 1e-12) if hp['loss_normalize'] else None
        st, m, meta = train_head(
            trX, trY, vaX, vaY, ust, args,
            lr=hp['lr'], epochs=args.tuned_epochs, schedule=hp['schedule'],
            label=f'Tuned {t} {r}/{args.tuned_trials}',
            hidden_dims=hdim, dropout=hp['dropout'], loss_scale=ls, seed=hp['seed'])
        record = {'element': t, 'run': r, **hp, 'val_median_mse': float(m), **meta}
        tres.append(record)
        if m < best_metric:
            best_metric, best_state, best_record = m, st, record.copy()
        if r == 1 or r % 25 == 0:
            print(f'tuned {t} {r}/{args.tuned_trials} val_median_mse={m:.4e} '
                  f'lr={hp["lr"]:.2e} sched={hp["schedule"]} norm={hp["loss_normalize"]}', flush=True)

    checkpoint = f'best_tuned_{t}.pt'
    torch.save(best_state, OUTPUT / checkpoint)
    best_tuned[t] = {**best_record, 'checkpoint': checkpoint, 'source_checkpoint': 'best_universal.pt'}
    # Preserve each finished element's records if a later element fails.
    pd.DataFrame(tres).to_csv(OUTPUT / 'tuned_records.csv', index=False)
    pd.DataFrame(best_tuned.values()).to_csv(OUTPUT / 'best_tuned_hyperparameters.csv', index=False)
    print(f'BEST TUNED {t}: run={best_record["run"]} '
          f'val_median_mse={best_metric:.4e} selection={best_record["selection"]}', flush=True)

summary = {
    'status': 'selected', 'selection_metric': 'validation_median_mse',
    'universal_selection_scope': 'pooled_validation_spectra',
    'tuned_selection_scope': 'per_element_validation_spectra',
    'features_dir': str(FEATURES_DIR.resolve()), 'feature_sha256': FEATURE_HASHES.copy(),
    'settings': vars(args),
    'best_universal': {'checkpoint': 'best_universal.pt', 'val_median_mse': float(um), **umeta, **uhyp},
    'best_tuned': best_tuned,
}
(OUTPUT / 'summary.json').write_text(_json.dumps(summary, indent=2))
display(pd.DataFrame(best_tuned.values())[
    ['element', 'run', 'lr', 'schedule', 'loss_normalize', 'seed',
     'best_epoch', 'selection', 'val_median_mse']
])


In [ ]:
def evaluate(head, task, data):
    X, y = data
    err = prediction_errors(head, X, y, args.batch_size)
    base = float(np.median(((y - train[task][1].mean(0)) ** 2).mean(1)))
    med = float(np.median(err))
    return {'mse': float(err.mean()), 'median_mse': med,
            'baseline_median_mse': base, 'eta': base / med if med > 0 else float('inf')}


def saved_head(checkpoint):
    state = torch.load(OUTPUT / checkpoint, map_location='cpu', weights_only=True)
    return make_head(hdim, hdrop, state)


if set(best_tuned) != set(TASKS):
    raise RuntimeError('Expected one selected tuned checkpoint per element')

uh = saved_head('best_universal.pt')
rows = []
# Check every saved winner BEFORE loading any test targets.
for t in TASKS:
    th = saved_head(best_tuned[t]['checkpoint'])
    universal_val = evaluate(uh, t, val[t])
    tuned_val = evaluate(th, t, val[t])
    expected_source = best_tuned[t]['source_val_median_mse']
    expected_tuned = best_tuned[t]['val_median_mse']
    if not np.isclose(universal_val['median_mse'], expected_source, rtol=1e-5, atol=1e-10):
        raise RuntimeError(f'{t}: Universal checkpoint does not match the sweep source')
    if not np.isclose(tuned_val['median_mse'], expected_tuned, rtol=1e-5, atol=1e-10):
        raise RuntimeError(f'{t}: saved tuned checkpoint does not match its selected validation score')
    if (tuned_val['median_mse'] > universal_val['median_mse']
            and not np.isclose(tuned_val['median_mse'], universal_val['median_mse'],
                               rtol=1e-5, atol=1e-10)):
        raise RuntimeError(f'{t}: selected tuned head is worse than its source on validation')
    for variant, metrics in (('UniversalXAS', universal_val), ('Tuned-UniversalXAS', tuned_val)):
        rows.append({'variant': variant, 'element': t, 'split': 'val', **metrics})

# Hyperparameters and checkpoints are now fixed. Evaluate test once per selected head.
for t in TASKS:
    test_data = load(t, 'test')
    th = saved_head(best_tuned[t]['checkpoint'])
    for variant, head in (('UniversalXAS', uh), ('Tuned-UniversalXAS', th)):
        rows.append({'variant': variant, 'element': t, 'split': 'test',
                     **evaluate(head, t, test_data)})

res = pd.DataFrame(rows)
res.to_csv(OUTPUT / 'final_evaluation.csv', index=False)

def settings_text(record):
    lr = record.get('lr')
    lr_text = '-' if lr is None else f'{float(lr):.1e}'
    schedule = record.get('schedule') or 'plateau'
    norm = record.get('loss_normalize')
    norm_text = '-' if norm is None else ('yes' if norm else 'no')
    return f'lr={lr_text}, {schedule}, norm={norm_text}, drop={float(record["dropout"]):.2f}, hid=500/500/550'

# All nine elements, with each element's own selected tuned checkpoint.
tbl = []
for t in ELEMENTS:
    task = t + '_FEFF'
    row = {'element': t, 'settings': f'U: {settings_text(uhyp)} | T: {settings_text(best_tuned[task])}'}
    for variant in ('UniversalXAS', 'Tuned-UniversalXAS'):
        for split in ('val', 'test'):
            m = res[(res.variant == variant) & (res.element == task) & (res.split == split)].iloc[0]
            row[variant + ' ' + split + ' median_mse'] = float(m.median_mse)
            row[variant + ' ' + split + ' eta'] = float(m.eta)
    tbl.append(row)
table = pd.DataFrame(tbl)
display(table.round(4))

def zn_showcase(split, data):
    task = 'Zn_FEFF'
    X, y = data
    tuned = saved_head(best_tuned[task]['checkpoint'])
    up = predict_spectra(uh, X, args.batch_size)
    tp = predict_spectra(tuned, X, args.batch_size)
    universal_error = ((up - y) ** 2).mean(1)
    percentiles = (10, 50, 90, 99)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
    for ax, percentile in zip(axes.flat, percentiles):
        target = np.percentile(universal_error, percentile)
        index = int(np.argmin(np.abs(universal_error - target)))
        ax.plot(y[index], color='black', lw=1.5, label='Target')
        ax.plot(up[index], color='tab:blue', lw=1.2, label='UniversalXAS')
        ax.plot(tp[index], color='tab:orange', lw=1.2, label='Tuned-UniversalXAS')
        ax.set_title(f'{percentile}th Universal error: {universal_error[index]:.2e}')
        ax.grid(alpha=0.2)
    axes[1, 0].set_xlabel('Energy-grid index'); axes[1, 1].set_xlabel('Energy-grid index')
    axes[0, 0].set_ylabel('Intensity'); axes[1, 0].set_ylabel('Intensity')
    axes[0, 0].legend(fontsize=9)
    fig.suptitle(f'Zn FEFF: {split} percentile showcase')
    fig.tight_layout()
    fig.savefig(OUTPUT / f'zn_percentile_showcase_{split}.png', dpi=180, bbox_inches='tight')
    display(fig); plt.close(fig)

zn_showcase('val', val['Zn_FEFF'])
zn_showcase('test', load('Zn_FEFF', 'test'))

summary['status'] = 'complete'
summary['feature_sha256'] = FEATURE_HASHES.copy()
(OUTPUT / 'summary.json').write_text(_json.dumps(summary, indent=2))
print('artifacts:', sorted(p.name for p in OUTPUT.iterdir()))
